### Introduction to layer Normalization 
1. Before going to the attention mechanism, each input embedding vectors should go to normalization 
2. Normalization helps to make the model more general and more suitable to make gradients stable during trainning phase. 
    - 2.1:- Helps us to tackle vanishing/ exploding gradient problem
3. Introduce co-variance in the layer, mean = 0 and variance = 1

In [1]:
import torch 
import torch.nn as nn

In [20]:
# Implementation of below example approach in class/modular format

class LayerNormalization(nn.Module):
    # constructor for this call 
    def __init__(self, feature_dims: int):
        # Super method to abstract nn.Module class 
        super().__init__()
        # Defining layer normalization parameters 
        self.eplison = 10e-5
        # Trainable scale parameter layer 
        self.scale = nn.Parameter(torch.zeros(feature_dims), requires_grad= True)
        # Trainable Step parameter layer 
        self.step = nn.Parameter(torch.ones(feature_dims), requires_grad= True)

    def forward(self, x: torch.tensor):
        # Calculation of layer Normalization (Target: mean = 0, variance = 1)
        layer_mean = torch.mean(x, keepdim= True, dim= -1)
        layer_var = torch.var(x, unbiased= False, keepdim= True, dim= -1)
        layer_normal = (x - layer_mean) / (layer_var - self.eplison) # division of eps- to avoid divison with 0

        return self.scale * layer_normal + self.step


In [21]:
# Checking implementation using obj of class 
inputs_embeddings = torch.tensor([
    [0.75, 0.98, 0.56, 0.78, 0.96], # The    
    [0.67, 0.75, 0.42, 0.52, 0.71], # Cat
    [0.81, 0.02, 0.35, 0.37, 0.87], # Sleeps 
])
feature_dims = inputs_embeddings.shape[-1] # 5 features dims 
batch = torch.stack((inputs_embeddings, inputs_embeddings), dim= 0)
layer_normalize = LayerNormalization(feature_dims)
print(f'LayerNormalization.Result: \n{layer_normalize(batch)}')

LayerNormalization.Result: 
tensor([[[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]],

        [[1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.],
         [1., 1., 1., 1., 1.]]], grad_fn=<AddBackward0>)


In [6]:
# Sample normalization on input embedding 
inputs_embeddings = torch.tensor([
    [0.75, 0.98, 0.56, 0.78, 0.96], # The    
    [0.67, 0.75, 0.42, 0.52, 0.71], # Cat
    [0.81, 0.02, 0.35, 0.37, 0.87], # Sleeps 
])

# calculating mean & variance on each column dimension
mean_sample = torch.mean(inputs_embeddings, dim= -1, keepdim= True)
variance_sample = torch.var(inputs_embeddings, dim= -1, keepdim= True, unbiased= False)
print(f'Sample.mean: \n{mean_sample}') 
print(f'Sample.variance: \n{variance_sample}')

# We need to have mean = 0 and variance = 1 for each token to have layer normalization 
layer_normal = (inputs_embeddings - mean_sample) / variance_sample**0.5
print(f"AfterNormalization.Values \n{layer_normal}")

# After normalization mean and variance to confirm 
mean_after = torch.mean(layer_normal, dim= -1, keepdim= True)
var_after = torch.var(layer_normal, dim= -1, keepdim= True, unbiased= False)
# values comming here are 10^n =~ 0 (not absolute 0 but very close to 0)
print(f'After.mean: \n{mean_after}') 
print(f'After.variance: \n{var_after}')

Sample.mean: 
tensor([[0.8060],
        [0.6140],
        [0.4840]])
Sample.variance: 
tensor([[0.0237],
        [0.0155],
        [0.1003]])
AfterNormalization.Values 
tensor([[-0.3640,  1.1311, -1.5992, -0.1690,  1.0011],
        [ 0.4503,  1.0936, -1.5601, -0.7559,  0.7720],
        [ 1.0293, -1.4651, -0.4231, -0.3600,  1.2188]])
After.mean: 
tensor([[ 3.9041e-07],
        [-2.0266e-07],
        [-8.3447e-08]])
After.variance: 
tensor([[1.0000],
        [1.0000],
        [1.0000]])
